# NCAA Player Atlas 2025-26
- Adapted from code written in 2024-25 season folder

In [16]:
## Dependencies

## System Libraries
import sys
import os
# Data handling
import pandas as pd
import geopandas as gpd
# Plotting and visualization
import matplotlib.pyplot as plt
from PIL import Image
## Map visualization
import folium
from folium.plugins import MarkerCluster
from folium.plugins import HeatMap
from folium.features import CustomIcon

### ROSTERFILE
# Path to the roster file
roster_path = os.path.join('..', 'data', 'player_info', 'roster_9-27-25_run_1_cleaned_20250927.csv')
roster_df = pd.read_csv(roster_path) # Load the roster file


############### NOT USED IN THIS SCRIPT ################
#### 2023 STATS FILE
# stats_path = os.path.join('..', 'data', 'player_stats_2023_v1.csv')
# stats_df = pd.read_csv(stats_path) # Load the stats file

# #### 2024 ROSTER FILE - need to recreate with code below
# ########## ROSTER SET WITH 2023 STATS FILE
# roster_stats_path = os.path.join('..', 'data', 'roster_2024_with_2023_stats.csv')
# roster_stats_df = pd.read_csv(roster_stats_path) # Load the roster file with stats


### SCHOOL INFO TABLE FOR LOGO PATHS
school_info_path = os.path.join('..', 'data', 'school_info', 'arena_school_info.csv')
school_info_df = pd.read_csv(school_info_path) # Load school info

# Path to logo folder
logo_folder = os.path.join('..', 'images', 'logos')

### SHAPEFILES
# Path to .geojson file with State Boundries
geojson_path = os.path.join('..', 'data', 'vault', 'combined-us-canada.geojson')
# Load the states shapefile
gdf_states = gpd.read_file(geojson_path)

# Path to shapefile with all US counties
shapefile_path = os.path.join('..', 'data', 'vault', 'cb_2018_us_county_500k.shp')
gdf = gpd.read_file(shapefile_path)
# Set the initial CRS (assuming it's in EPSG:4326, but you may need to verify the original CRS)
gdf = gdf.set_crs(epsg=4326)

## CHECK SHAPEFILES FOR COMPATIBILITY
# Set the CRS for both dataframes if it's missing
if gdf.crs is None:
    gdf.set_crs(epsg=4326, inplace=True)  # Assuming coordinates are in WGS 84 (lat/lon)

if gdf_states.crs is None:
    gdf_states.set_crs(epsg=4326, inplace=True)  # Assuming coordinates are in WGS 84 (lat/lon)



# Check the first few rows of the DataFrames
# roster_df.head()
# gdf_states.head()
# gdf.head()
school_info_df.head()
# roster_stats_df.head()



,Team,Arena,Capacity,Sheet_length,Sheet_width,School,Latitude,Longitude,hex1,hex2,hex3,simp_color,logo_abv,abv,ncaa_name,ncaa_data_alts,eliteprospects_url
0,Air Force,Cadet Ice Arena,2470,200,85,Air Force,39.013739,-104.883727,3087,8a8d8f,NaN,NaN,afa,Air Force,Air Force,"AIRFOR, Air Force",https://www.eliteprospects.com/team/2453/air-f...
1,Alaska,Carlson Center,4595,200,100,Alaska,64.842124,-147.763841,236192,ffcd00,NaN,NaN,akf,Alaska,Alas Fairbanks,"AK FBK, Alas. Fairbanks",https://www.eliteprospects.com/team/2071/univ....
2,Alaska Anchorage,Avis Alaska Sports Complex,800,200,85,Alaska-Anchorage,61.205536,-149.872737,00583d,ffc425,NaN,NaN,aka,UAA,Alas Anchorage,"AK ANC, Alas. Anchorage",https://www.eliteprospects.com/team/1915/univ....
3,American Intl,MassMutual Center,6866,200,85,American Int'l,42.118003,-72.554326,0,ffb60f,NaN,NaN,aic,AIC,American Intl,"AM INT, American Int'l",https://www.eliteprospects.com/team/1252/ameri...
4,American Int'l,MassMutual Center,6866,200,85,American Int'l,42.118003,-72.554326,0,ffb60f,NaN,NaN,aic,AIC,American Intl,"AM INT, American Int'l",https://www.eliteprospects.com/team/1252/ameri...


In [17]:
## Merge 2024-25 Stats with 2025 Roster

## Path to 2024-25 Stats File
stats_2024_path = os.path.join('..', 'data', 'player_info', 'NCAA_stats_2024-25.csv')
stats_2024_df = pd.read_csv(stats_2024_path) # Load the stats file

# Rename Player name Column to match roster_df
stats_2024_df.rename(columns={'Clean_Player':'Player'}, inplace=True)
# Display the first few rows to understand its structure
stats_2024_df.head()
# roster_df.head()

,Player,Team,G,A,Pts,plus_minus,Sh,TOI_sec,PIM,FOW,FOL,Games_Played,FO%,TOI
0,A.J. Hodges,Bentley,9,9,18,7,78,28504.0,8,19.0,22.0,30,46.341463,07:55:04
1,A.J. Macaulay,Bemidji State,0,2,2,-5,17,22475.0,4,0.0,0.0,28,NaN,06:14:35
2,AJ Casperson,Long Island,0,0,0,0,1,249.0,0,0.0,0.0,1,NaN,00:04:09
3,Aaron Bohlinger,Quinnipiac,3,11,14,11,34,40704.0,4,0.0,0.0,32,NaN,11:18:24
4,Aaron Grounds,Long Island,0,0,0,0,0,544.0,0,0.0,0.0,1,NaN,00:09:04


In [18]:
roster_df.head()

,Current Team,Last_Name,First_Name,No,Position,Yr,Ht,Wt,DOB,Hometown,Height_Inches,Draft_Year,NHL_Team,D_Round,Last Team,League,City,State_Province,Country
0,Michigan,Barnett,Asher,4,Defensemen,Fr,6-1,197,5/16/2007,"Wilmette, Ill.",73,2025.0,EDM,5.0,USA U-18 Team,USHL,Wilmette,Illinois,USA
1,Michigan,Duke,Tyler,5,Defensemen,Fr,5-10,194,7/19/2004,"Strongsville, Ohio",70,NaN,NaN,NaN,USA U-18 Team,NTDP,Strongsville,Ohio,USA
2,Michigan,Fantilli,Luca,63,Defensemen,Sr,6-0,183,12/30/2002,"Kleinburg, Ont.",72,NaN,NaN,NaN,Chicago,USHL,Kleinburg,Ontario,Canada
3,Michigan,Gust,Miles,23,Defensemen,So,5-9,181,3/17/2004,"Chicago, Ill.",69,NaN,NaN,NaN,Muskegon,USHL,Chicago,Illinois,USA
4,Michigan,Hady,Hunter,3,Defensemen,So,6-4,198,8/2/2004,"Rochester, Minn.",76,NaN,NaN,NaN,Chicago,USHL,Rochester,Minnesota,USA


In [19]:
# Show all rows in stats_2004_df that Team is RPI or Rensselaer
stats_2024_df[(stats_2024_df['Team'] == 'RPI')]

roster_df[(roster_df['Current Team'] == 'RPI')]

# Replace RPI with Rensselaer in roster_df 'Current Team' column
roster_df['Current Team'] = roster_df['Current Team'].replace('RPI', 'Rensselaer')

In [20]:

roster_df.rename(columns={'Current Team':'Team'}, inplace=True) # Rename 'Current_Team' to 'Team' for consistency

# Create a new 'Player' column that combines the player's first and last name
roster_df['Player'] = roster_df['First_Name'] + ' ' + roster_df['Last_Name']

roster_df['Player'] = roster_df['Player'].str.strip() # Strip any leading or trailing white space

roster_df.head()

,Team,Last_Name,First_Name,No,Position,Yr,Ht,Wt,DOB,Hometown,Height_Inches,Draft_Year,NHL_Team,D_Round,Last Team,League,City,State_Province,Country,Player
0,Michigan,Barnett,Asher,4,Defensemen,Fr,6-1,197,5/16/2007,"Wilmette, Ill.",73,2025.0,EDM,5.0,USA U-18 Team,USHL,Wilmette,Illinois,USA,Asher Barnett
1,Michigan,Duke,Tyler,5,Defensemen,Fr,5-10,194,7/19/2004,"Strongsville, Ohio",70,NaN,NaN,NaN,USA U-18 Team,NTDP,Strongsville,Ohio,USA,Tyler Duke
2,Michigan,Fantilli,Luca,63,Defensemen,Sr,6-0,183,12/30/2002,"Kleinburg, Ont.",72,NaN,NaN,NaN,Chicago,USHL,Kleinburg,Ontario,Canada,Luca Fantilli
3,Michigan,Gust,Miles,23,Defensemen,So,5-9,181,3/17/2004,"Chicago, Ill.",69,NaN,NaN,NaN,Muskegon,USHL,Chicago,Illinois,USA,Miles Gust
4,Michigan,Hady,Hunter,3,Defensemen,So,6-4,198,8/2/2004,"Rochester, Minn.",76,NaN,NaN,NaN,Chicago,USHL,Rochester,Minnesota,USA,Hunter Hady


In [21]:
## First merge Attempt - Just on Player because of transfers
merged_df = pd.merge(roster_df, stats_2024_df, on='Player', how='left', suffixes=('_roster', '_stats'))
merged_df.head()

,Team_roster,Last_Name,First_Name,No,Position,Yr,Ht,Wt,DOB,Hometown,...,Pts,plus_minus,Sh,TOI_sec,PIM,FOW,FOL,Games_Played,FO%,TOI
0,Michigan,Barnett,Asher,4,Defensemen,Fr,6-1,197,5/16/2007,"Wilmette, Ill.",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Michigan,Duke,Tyler,5,Defensemen,Fr,5-10,194,7/19/2004,"Strongsville, Ohio",...,13.0,-2.0,37.0,34161.0,33.0,0.0,0.0,29.0,NaN,09:29:21
2,Michigan,Fantilli,Luca,63,Defensemen,Sr,6-0,183,12/30/2002,"Kleinburg, Ont.",...,6.0,-5.0,30.0,37880.0,8.0,0.0,0.0,34.0,NaN,10:31:20
3,Michigan,Gust,Miles,23,Defensemen,So,5-9,181,3/17/2004,"Chicago, Ill.",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Michigan,Hady,Hunter,3,Defensemen,So,6-4,198,8/2/2004,"Rochester, Minn.",...,1.0,-2.0,10.0,18495.0,35.0,0.0,0.0,32.0,NaN,05:08:15


## Get Geocode Locations for all hometowns in the dataset
- Using Google Maps API

In [22]:
# Group by the location columns and count the number of players in each group
location_summary = merged_df.groupby(['City', 'State_Province', 'Country']).size().reset_index(name='Player_Count')

# Display the resulting DataFrame
print(location_summary)

                 City    State_Province   Country  Player_Count
0          Abbotsford  British Columbia    Canada             5
1            Aberdeen      Saskatchewan    Canada             1
2            Aberdeen          Scotland  Scotland             1
3            Abington     Massachusetts       USA             1
4             Airdrie           Alberta    Canada             2
..                ...               ...       ...           ...
917  Yorktown Heights          New York       USA             1
918            Zagreb           Croatia   Croatia             1
919            Zilina          Slovakia  Slovakia             1
920        Zionsville           Indiana       USA             1
921            Örebro            Sweden    Sweden             1

[922 rows x 4 columns]


In [23]:
# ### GEOCODING USING GOOGLE MAPS API 
# # LCHECK FOR AND LOAD GEOCODED DATA BEFORE RUNNING - THIS COSTS MONEY

# import googlemaps
# import pandas as pd
# import config




# # Initialize the Google Places API client
# gmaps = googlemaps.Client(key=config.g_key)

# def geocode_google_places(row):
#     try:
#         location_str = f"{row['City']}, {row['State_Province']}, {row['Country']}"
#         print(f"Querying location: {location_str}")  # Debugging output
#         geocode_result = gmaps.geocode(location_str)
        
#         # Check the API response
#         print(f"Geocode result: {geocode_result}")  # Debugging output
        
#         # Check if we got a valid result
#         if geocode_result and 'geometry' in geocode_result[0]:
#             location = geocode_result[0]['geometry']['location']
#             return pd.Series([location['lat'], location['lng']])
#         else:
#             return pd.Series([None, None])  # Return None if no valid result
#     except Exception as e:
#         print(f"Error encountered: {e}")  # Debugging output
#         return pd.Series([None, None])  # Handle errors gracefully

# # Apply the geocode function to the data using Google Places API
# location_summary[['Latitude', 'Longitude']] = location_summary.apply(geocode_google_places, axis=1)

# # Filter out rows with missing coordinates if needed
# location_summary_transformed = location_summary.dropna(subset=['Latitude', 'Longitude'])

# # Display the cleaned data with coordinates
# location_summary_transformed.head()

In [24]:
# ## Save the geocoded data to a CSV file to avoid re-running the API
# DATA_FOLDER = os.path.join('..', 'data', 'player_info')
# # location_summary_transformed.to_csv(DATA_FOLDER + '2025_tourney_location_summary_geocoded.csv', index=False)
# location_summary_transformed.to_csv(DATA_FOLDER + '2025_full_roster_location_summary_geocoded.csv', index=False)

In [25]:


## Load the geocoded data from the CSV file
# location_summary_geocoded = pd.read_csv(DATA_FOLDER + '2025_tourney_location_summary_geocoded.csv')
location_summary_geocoded = pd.read_csv('../data/full_roster_location_summary_geocoded_2025.csv')
location_summary_geocoded.head()

,City,State_Province,Country,Player_Count,Latitude,Longitude
0,Abbotsford,British Columbia,Canada,5,49.050438,-122.304470
1,Aberdeen,Saskatchewan,Canada,1,52.326104,-106.291491
2,Aberdeen,Scotland,Scotland,1,57.149889,-2.093753
3,Abington,Massachusetts,USA,1,42.104823,-70.945322
4,Airdrie,Alberta,Canada,2,51.292697,-114.013411


In [26]:
### MERGE THE LATITUDE AND LONGITUDE DATA WITH THE ROSTER DATA
roster_ncaa_ytd_location = pd.merge(merged_df, location_summary_geocoded, on=['City', 'State_Province', 'Country'], how='left')

### HOTFIX FOR Czech Republic ISSUE - NEED TO REPLACE WITH CZECHIA
roster_ncaa_ytd_location['Country'] = roster_ncaa_ytd_location['Country'].replace('Czech Republic', 'Czechia')
# In State_Province column, replace 'Czech Republic' with 'Czechia'
roster_ncaa_ytd_location['State_Province'] = roster_ncaa_ytd_location['State_Province'].replace('Czech Republic', 'Czechia')

# Change Scotland to United Kingdom
roster_ncaa_ytd_location['Country'] = roster_ncaa_ytd_location['Country'].replace('Scotland', 'United Kingdom')
roster_ncaa_ytd_location['State_Province'] = roster_ncaa_ytd_location['State_Province'].replace('Scotland', 'United Kingdom')


# Check the merged data
# print(roster_ncaa_ytd_location.info())
roster_ncaa_ytd_location.head()



,Team_roster,Last_Name,First_Name,No,Position,Yr,Ht,Wt,DOB,Hometown,...,TOI_sec,PIM,FOW,FOL,Games_Played,FO%,TOI,Player_Count,Latitude,Longitude
0,Michigan,Barnett,Asher,4,Defensemen,Fr,6-1,197,5/16/2007,"Wilmette, Ill.",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,42.072251,-87.722838
1,Michigan,Duke,Tyler,5,Defensemen,Fr,5-10,194,7/19/2004,"Strongsville, Ohio",...,34161.0,33.0,0.0,0.0,29.0,NaN,09:29:21,2.0,41.314497,-81.835690
2,Michigan,Fantilli,Luca,63,Defensemen,Sr,6-0,183,12/30/2002,"Kleinburg, Ont.",...,37880.0,8.0,0.0,0.0,34.0,NaN,10:31:20,2.0,43.838502,-79.623541
3,Michigan,Gust,Miles,23,Defensemen,So,5-9,181,3/17/2004,"Chicago, Ill.",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,41.883250,-87.632388
4,Michigan,Hady,Hunter,3,Defensemen,So,6-4,198,8/2/2004,"Rochester, Minn.",...,18495.0,35.0,0.0,0.0,32.0,NaN,05:08:15,4.0,44.019329,-92.458833


## Talley the Stats by State (Total Games_Played, Goals, Assist, PIM, ect)

In [27]:
# Group by State_Province and add all up player stats to get a state by state summary
state_summary = roster_ncaa_ytd_location.groupby('State_Province').agg({
    'G': 'sum',
    'A': 'sum',
    'Pts': 'sum',
    'plus_minus': 'sum',
    'Sh': 'sum',
    'TOI_sec': 'sum',
    'PIM': 'sum',
    'Games_Played': 'sum'
}).reset_index()

# Calculate the average stats per game for each state
state_summary['GPG'] = state_summary['G'] / state_summary['Games_Played']
state_summary['APG'] = state_summary['A'] / state_summary['Games_Played']
state_summary['PtsPG'] = state_summary['Pts'] / state_summary['Games_Played']
state_summary['plus_minus_PG'] = state_summary['plus_minus'] / state_summary['Games_Played']
state_summary['ShPG'] = state_summary['Sh'] / state_summary['Games_Played']
state_summary['TOI_secPG'] = state_summary['TOI_sec'] / state_summary['Games_Played']
state_summary['PIMPG'] = state_summary['PIM'] / state_summary['Games_Played']

# Calculate the number of players per state
state_summary['Player_Count'] = roster_ncaa_ytd_location.groupby('State_Province').size().values
# Calculate average stats per player for each state
state_summary['Games_PlayedPP'] = state_summary['Games_Played'] / state_summary['Player_Count']
state_summary['GPP'] = state_summary['G'] / state_summary['Player_Count']
state_summary['APP'] = state_summary['A'] / state_summary['Player_Count']
state_summary['PtsPP'] = state_summary['Pts'] / state_summary['Player_Count']
state_summary['plus_minus_PP'] = state_summary['plus_minus'] / state_summary['Player_Count']
state_summary['ShPP'] = state_summary['Sh'] / state_summary['Player_Count']
state_summary['TOI_secPP'] = state_summary['TOI_sec'] / state_summary['Player_Count']
state_summary['PIMPP'] = state_summary['PIM'] / state_summary['Player_Count']


# Display the resulting DataFrame
state_summary.head()

,State_Province,G,A,Pts,plus_minus,Sh,TOI_sec,PIM,Games_Played,GPG,...,PIMPG,Player_Count,Games_PlayedPP,GPP,APP,PtsPP,plus_minus_PP,ShPP,TOI_secPP,PIMPP
0,Alaska,42.0,81.0,123.0,-13.0,409.0,264475.0,95.0,272.0,0.154412,...,0.349265,18,15.111111,2.333333,4.500000,6.833333,-0.722222,22.722222,14693.055556,5.277778
1,Alberta,263.0,408.0,671.0,-75.0,2742.0,1731352.0,825.0,2002.0,0.131369,...,0.412088,117,17.111111,2.247863,3.487179,5.735043,-0.641026,23.435897,14797.880342,7.051282
2,Arizona,26.0,48.0,74.0,4.0,221.0,148298.0,103.0,153.0,0.169935,...,0.673203,9,17.000000,2.888889,5.333333,8.222222,0.444444,24.555556,16477.555556,11.444444
3,Austria,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,...,NaN,3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,Belarus,1.0,2.0,3.0,3.0,27.0,17957.0,20.0,28.0,0.035714,...,0.714286,3,9.333333,0.333333,0.666667,1.000000,1.000000,9.000000,5985.666667,6.666667


In [28]:
## Export to csv to check in excel
# state_summary.to_csv(TEMP_FOLDER + '2025_tourney_state_summary.csv', index=False)

## Output Breakdown Tables: Hometown and First Name

### Breakdown by Hometown

In [29]:
roster_ncaa_ytd_location.head()

,Team_roster,Last_Name,First_Name,No,Position,Yr,Ht,Wt,DOB,Hometown,...,TOI_sec,PIM,FOW,FOL,Games_Played,FO%,TOI,Player_Count,Latitude,Longitude
0,Michigan,Barnett,Asher,4,Defensemen,Fr,6-1,197,5/16/2007,"Wilmette, Ill.",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,42.072251,-87.722838
1,Michigan,Duke,Tyler,5,Defensemen,Fr,5-10,194,7/19/2004,"Strongsville, Ohio",...,34161.0,33.0,0.0,0.0,29.0,NaN,09:29:21,2.0,41.314497,-81.835690
2,Michigan,Fantilli,Luca,63,Defensemen,Sr,6-0,183,12/30/2002,"Kleinburg, Ont.",...,37880.0,8.0,0.0,0.0,34.0,NaN,10:31:20,2.0,43.838502,-79.623541
3,Michigan,Gust,Miles,23,Defensemen,So,5-9,181,3/17/2004,"Chicago, Ill.",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,41.883250,-87.632388
4,Michigan,Hady,Hunter,3,Defensemen,So,6-4,198,8/2/2004,"Rochester, Minn.",...,18495.0,35.0,0.0,0.0,32.0,NaN,05:08:15,4.0,44.019329,-92.458833


In [30]:
# Copy Team_roster to new column called Team
roster_ncaa_ytd_location['Team'] = roster_ncaa_ytd_location['Team_roster']

# ---- 1) GROUP BY CITY, STATE/PROVINCE, COUNTRY ----
grouped_city = (
    roster_ncaa_ytd_location
    .groupby(["City", "State_Province", "Country"], dropna=False)
    .agg(
        Count=("Player", "size"),  # number of players from that city
        Teams=("Team", lambda x: ", ".join(sorted(set(x))))  # unique teams
    )
    .reset_index()
)

# Sort by Count descending to see which city has the most players
grouped_city = grouped_city.sort_values(by="Count", ascending=False)

# (Optional) Keep the top 10
top_10_cities = grouped_city.head(10)

# Print or examine the result
print(top_10_cities)

                City    State_Province Country  Count  \
111          Calgary           Alberta  Canada     48   
815          Toronto           Ontario  Canada     34   
489      Mississauga           Ontario  Canada     15   
775        Stockholm            Sweden  Sweden     15   
497         Montréal            Quebec  Canada     14   
759        St. Louis          Missouri     USA     14   
223         Edmonton           Alberta  Canada     14   
551  North Vancouver  British Columbia  Canada     13   
561         Oakville           Ontario  Canada     12   
582           Ottawa           Ontario  Canada     12   

                                                 Teams  
111  Air Force, Alaska, Alaska Anchorage, Arizona S...  
815  Alaska Anchorage, Bentley, Boston University, ...  
489  Boston College, Canisius, Dartmouth, Harvard, ...  
775  Bentley, Boston University, Canisius, Colorado...  
497  Brown, Canisius, Clarkson, Denver, Lake Superi...  
759  Augustana, Cornell, Ferri

### Breakdown by First Name

In [31]:
# roster_ncaa_ytd_location.info()

# Strip any leading or trailing whitespace from the 'First_Name' and 'Last_Name' columns
roster_ncaa_ytd_location['First_Name'] = roster_ncaa_ytd_location['First_Name'].str.strip()
roster_ncaa_ytd_location['Last_Name'] = roster_ncaa_ytd_location['Last_Name'].str.strip()

# ---- 2) GROUP BY FIRST NAME ----
grouped_names = (
    roster_ncaa_ytd_location
    .groupby("First_Name", dropna=False)
    .agg(
        NumPlayers=("First_Name", "size"),       # total players with that first name
        NumTeams=("Team", "nunique"),           # how many distinct teams
        Teams=("Team", lambda x: ", ".join(sorted(set(x)))),
    )
    .reset_index()
)

# Sort by the number of players descending
grouped_names = grouped_names.sort_values(by="NumPlayers", ascending=False)

# (Optional) Take top 10
top_10_names = grouped_names.head(10)

# Print or examine the result
print(top_10_names)

    First_Name  NumPlayers  NumTeams  \
274       Jack          47        33   
521       Ryan          42        29   
593      Tyler          28        24   
397       Luke          27        23   
435    Michael          26        23   
424    Matthew          23        16   
283       Jake          22        18   
473       Owen          21        18   
425        Max          21        16   
276      Jacob          21        17   

                                                 Teams  
274  Arizona State, Army, Bentley, Boston Universit...  
521  Alaska Anchorage, Bemidji State, Bentley, Bost...  
593  Alaska Anchorage, Augustana, Bowling Green, Br...  
397  Alaska Anchorage, Bemidji State, Canisius, Cla...  
435  Air Force, Alaska, Bentley, Boston College, Br...  
424  Brown, Clarkson, Harvard, Holy Cross, Massachu...  
283  Air Force, Bemidji State, Bentley, Boston Coll...  
473  Army, Augustana, Bentley, Boston University, B...  
425  Bemidji State, Boston University, Colgate

## Create the Map Using Follium

### Get top scorer name and statline for each state

In [87]:
            # Top Scorer: <strong>{row['TopScorer_Name']}</strong><br>
            # {row['TopScorer_StatLine']}

In [88]:
gdf_states.head()
roster_df.head()
roster_ncaa_ytd_location.head()
# renam Team_roster to Team
roster_ncaa_ytd_location.rename(columns={'Team_roster':'Team'}, inplace=True)

In [89]:
# TopScorer_Name - column name for the top scorer's name
# TopScorer_StatLine - column name for the top scorer's stat line
#### statline will be string such as "20 G, 30 A, 50 Pts, +10, 100 SOG"

roster_df = roster_ncaa_ytd_location
    
# Append a suffix to create a unique key for NA states/provinces.
gdf_states['State_Province'] = gdf_states['name'] + " (NA)"
na_states = set(gdf_states['State_Province'])

# 2. Update the State Province Names to work with something down the line
def update_state_name(x):
    # If the NA version exists, use it; otherwise, leave unchanged.
    if (x + " (NA)") in na_states:
        return x + " (NA)"
    else:
        return x

# Find nan values in the State_Province column and fill with empty string
roster_df['State_Province'] = roster_df['State_Province'].fillna('')
# find '[nan]' strings and replace with empty string
roster_df['State_Province'] = roster_df['State_Province'].replace('nan', '')
# Make sure State_Province column is string type
roster_df['State_Province'] = roster_df['State_Province'].astype(str)
# Update the State_Province column with the modified names
roster_df["State_Province_Mod"] = roster_df["State_Province"].apply(update_state_name)
# Copy the State_Province_Mod column back to the State_Province column ### ATTEMPTED HOTFIX
roster_df["State_Province"] = roster_df["State_Province_Mod"]


idx = roster_df.groupby("State_Province_Mod")["Pts"].idxmax()
idx = idx.dropna().astype(int)

# Use those indices to create a DataFrame of top scoring players per territory.
top_scorers = roster_df.loc[idx].copy()

# Helper function to format the PlusMinus stat with a plus sign if positive.
def format_plusminus(pm):
    try:
        return f"+{pm}" if pm > 0 else f"{pm}"
    except Exception:
        return str(pm)

# Create the TopScorer_StatLine for each player.
top_scorers["TopScorer_StatLine"] = top_scorers.apply(
    lambda row: f"{row['G']} G, {row['A']} A, {row['Pts']} Pts, {format_plusminus(row['plus_minus'])}, {row['Sh']} SOG",
    axis=1
)

# Rename the player's name column to TopScorer_Name.
top_scorers.rename(columns={"Player": "TopScorer_Name"}, inplace=True)

# Create the final DataFrame with the columns you need.
top_scorer_result_df = top_scorers[["State_Province_Mod", "Team", "Hometown", "City", "TopScorer_Name", "TopScorer_StatLine"]].copy()

# Create a new column called 'State_Province' (copy of 'State_Province_Mod') to match the expected output.
top_scorer_result_df['State_Province'] = top_scorer_result_df['State_Province_Mod']

# Optional: Display the first few rows to verify the results.
print(top_scorer_result_df.head())




         State_Province_Mod          Team           Hometown        City  \
946             Alaska (NA)    Penn State  Fairbanks, Alaska   Fairbanks   
953            Alberta (NA)    Penn State      Calgary, Alb.     Calgary   
663            Arizona (NA)  North Dakota     Phoenix, Ariz.     Phoenix   
299                 Belarus        Alaska     Minsk, Belarus       Minsk   
1383  British Columbia (NA)   Connecticut   Port Moody, B.C.  Port Moody   

          TopScorer_Name                          TopScorer_StatLine  \
946         Mac Gadowsky    16.0 G, 26.0 A, 42.0 Pts, +5.0, 99.0 SOG   
953           Aiden Fink  23.0 G, 30.0 A, 53.0 Pts, +15.0, 144.0 SOG   
663      Jake Livanavage      4.0 G, 24.0 A, 28.0 Pts, 0.0, 59.0 SOG   
299   Fyodor Nikolayenya       1.0 G, 2.0 A, 3.0 Pts, +3.0, 27.0 SOG   
1383         Ryan Tattle  18.0 G, 14.0 A, 32.0 Pts, +10.0, 118.0 SOG   

             State_Province  
946             Alaska (NA)  
953            Alberta (NA)  
663            Arizo

In [90]:
import numpy as np  # add this at the top with the other imports

def _clean_coords(df):
    """Coerce Latitude/Longitude to numeric and drop invalid coordinates."""
    df = df.copy()
    df['Latitude']  = pd.to_numeric(df['Latitude'], errors='coerce')
    df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

    valid = (
        df['Latitude'].between(-90, 90) &
        df['Longitude'].between(-180, 180)
    )
    df = df[valid].dropna(subset=['Latitude', 'Longitude'])
    # also drop (0,0) if it sneaks in
    df = df[~((df['Latitude'] == 0) & (df['Longitude'] == 0))]
    return df

    
# Function to apply a circular offset to markers with the same location
def add_circular_offset(lat, lon, count, index, radius=0.007):
    """
    Distributes markers in a circular pattern around a central point.
    The radius increases with the number of markers to prevent overlap.
    
    Modified to produce a larger spread:
    Instead of scaling the radius with (1 + count/4),
    we now scale it with (1 + count/2) for a wider spread.
    """
    # Calculate angle for the current marker (in degrees and then convert to radians)
    angle = (360 / count) * index
    radians = math.radians(angle)
    
    # Increase the radius scaling factor to spread markers further apart.
    dynamic_radius = radius * (1 + (count / 2))
    
    # Calculate offsets for latitude and longitude using circular placement
    lat_offset = lat + (dynamic_radius * math.cos(radians))
    lon_offset = lon + (dynamic_radius * math.sin(radians))
    
    return lat_offset, lon_offset



In [91]:
roster_ncaa_ytd_location.head()

# Show any Records with RPI or Rensselaer 
roster_ncaa_ytd_location[(roster_ncaa_ytd_location['Team'] == 'RPI') | (roster_ncaa_ytd_location['Team'] == 'Rensselaer')]
roster_ncaa_ytd_location.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1763 entries, 0 to 1762
Data columns (total 37 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Team                1763 non-null   object 
 1   Last_Name           1763 non-null   object 
 2   First_Name          1763 non-null   object 
 3   No                  1763 non-null   int64  
 4   Position            1763 non-null   object 
 5   Yr                  1756 non-null   object 
 6   Ht                  1763 non-null   object 
 7   Wt                  1763 non-null   int64  
 8   DOB                 1759 non-null   object 
 9   Hometown            1759 non-null   object 
 10  Height_Inches       1763 non-null   int64  
 11  Draft_Year          247 non-null    float64
 12  NHL_Team            247 non-null    object 
 13  D_Round             247 non-null    float64
 14  Last Team           1762 non-null   object 
 15  League              1738 non-null   object 
 16  City  

In [92]:
import os
import math
import re
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import HeatMap, MarkerCluster
from folium.features import CustomIcon
from markupsafe import Markup

# --- Assume these variables are defined elsewhere:
# roster_ncaa_ytd_location, school_info_df, logo_folder, gdf_states, state_summary, top_scorer_result_df, add_circular_offset

# Rename the data to merged_df
merged_df = roster_ncaa_ytd_location.copy()

# Assign unique index per player in each city group
merged_df['city_group_index'] = merged_df.groupby(['City', 'State_Province', 'Country']).cumcount()

# Assign 'Player_Count' per city directly using transform
merged_df['Player_Count'] = merged_df.groupby(['City', 'State_Province', 'Country'])['First_Name'].transform('count')

# Set Logo Size (width, height in pixels)
logo_size = (55, 55)  # Adjust as needed

# Convert number columns to integer type
int_columns = ['No', 'Height_Inches', 'Wt', 'Draft_Year', 'D_Round',
               'G', 'A', 'Pts', 'plus_minus', 'Sh', 'PIM', 'Games_Played']
for col in int_columns:
    merged_df[col] = merged_df[col].astype('Int64')


def create_map_with_team_logos(merged_df, school_info_df, logo_folder, gdf_states,
                               map_center=[45.0, -93.0], zoom_start=4):
    """
    Build an interactive map with multiple layers:
      - Base tile layers (light theme used here)
      - Choropleth shading of territories
      - Dynamic label markers for player counts
      - GeoJSON layer with rich HTML tooltips
      - Heatmap and marker clusters for individual player markers
      - Custom CSS for styling
    """
    # 1. Initialize the base map and add tile layers.
    folium_map = folium.Map(location=map_center, zoom_start=zoom_start,
                            tiles='OpenStreetMap', name='Default Map')
    folium.TileLayer('CartoDB positron', name='Light Theme', attr=".").add_to(folium_map)
    # Optional: add a dark theme if desired.
    # folium.TileLayer('CartoDB dark_matter', name='Dark Theme', attr=".").add_to(folium_map)

    # Inject custom CSS to load the Exo 2 font and style tooltips.
    font_link = """
    <link href="https://fonts.googleapis.com/css2?family=Exo+2:wght@400;700&display=swap" rel="stylesheet">
    """
    custom_font_css = """
    <style>
        .leaflet-tooltip, .folium-tooltip { 
            font-family: 'Exo 2', sans-serif; 
            font-size: 12px; 
        }
    </style>
    """
    folium_map.get_root().header.add_child(folium.Element(font_link + custom_font_css))

    # 2. Prepare GeoDataFrames for territories.
    # Append suffix for detailed NA states.
    gdf_states['State_Province'] = gdf_states['name'] + " (NA)"
    na_states = set(gdf_states['State_Province'])

    # Update merged_df so NA territories use the disambiguated key.
    def update_state_name(x):
        return x + " (NA)" if (x + " (NA)") in na_states else x
    merged_df['State_Province_Mod'] = merged_df['State_Province'].apply(update_state_name)

    # Load global boundaries.
    gdf_global = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    na_country_names = ['United States of America', 'Canada']
    gdf_global_non_na = gdf_global[~gdf_global['name'].isin(na_country_names)].copy()
    gdf_global_non_na['State_Province'] = gdf_global_non_na['name']

    # Combine NA detailed data with global data.
    gdf_combined = pd.concat([gdf_states, gdf_global_non_na], ignore_index=True)

    # 3. Build the choropleth layer based on player counts.
    territory_counts = merged_df['State_Province_Mod'].value_counts()
    territory_counts_df = pd.DataFrame(territory_counts).reset_index()
    territory_counts_df.columns = ['State_Province', 'Player_Count']

    custom_bins = [0, 1, 5, 10, 15, 25, 50, 75, 100, 150, 175, 200, 225, 250]
    geojson_data = gdf_combined.__geo_interface__

    folium.Choropleth(
        geo_data=geojson_data,
        data=territory_counts_df,
        columns=['State_Province', 'Player_Count'],
        key_on='feature.properties.State_Province',
        fill_color='YlGn',
        fill_opacity=0.5,
        line_opacity=0.2,
        legend_name='Number of Players by Territory',
        bins=custom_bins,
        reset=True,
        name='Shade by Player Count'
    ).add_to(folium_map)

    # 4. Add dynamic label markers showing player counts.
    gdf_combined_subset = gdf_combined[['State_Province', 'geometry']]
    territory_counts_gdf = gdf_combined_subset.merge(territory_counts_df,
                                                     on='State_Province', how='left')
    territory_counts_gdf['centroid'] = territory_counts_gdf.geometry.centroid

    labels_layer = folium.FeatureGroup(name='Player Count by Territory')
    for idx, row in territory_counts_gdf.iterrows():
        if pd.notnull(row['Player_Count']):
            lat, lon = row['centroid'].y, row['centroid'].x
            player_count = int(row['Player_Count'])
            label = folium.Marker(
                location=[lat, lon],
                icon=folium.DivIcon(
                    html=f"""
                    <div class="dynamic-label" style="
                        font-family: 'Exo 2', sans-serif;
                        font-weight: bold;
                        font-size: 16px;
                        color: black;
                        text-align: center;
                        padding: 2px;
                    ">
                        {player_count}
                    </div>
                    """
                )
            )
            labels_layer.add_child(label)
    labels_layer.add_to(folium_map)

    # Add custom JavaScript for dynamic font sizing on zoom.
    dynamic_font_script = """
    <script>
        function updateLabelSizes() {
            var zoom = map.getZoom();
            var newFontSize = Math.max(12, Math.min(30, zoom * 2));
            var labels = document.getElementsByClassName('dynamic-label');
            for (var i = 0; i < labels.length; i++) {
                labels[i].style.fontSize = newFontSize + 'px';
            }
        }
        map.on('zoomend', updateLabelSizes);
        updateLabelSizes();
    </script>
    """
    folium_map.get_root().html.add_child(folium.Element(dynamic_font_script))

    # 5. Merge summary stats and top scorer info for tooltips.
    # Update state_summary keys for NA territories.
    na_original_names = set(gdf_states['name'])
    def add_na_suffix(x):
        return x + " (NA)" if x in na_original_names else x
    state_summary['State_Province'] = state_summary['State_Province'].apply(add_na_suffix)

    # Merge summary stats and top scorer info with the combined GeoDataFrame.
    gdf_tooltip = gdf_combined.merge(state_summary, on="State_Province", how="left")
    gdf_tooltip = gdf_tooltip.merge(top_scorer_result_df, on="State_Province", how="left")

    # Convert TOI_sec to a Timedelta and create a formatted string.
    gdf_tooltip['TOI_timedelta'] = pd.to_timedelta(gdf_tooltip['TOI_sec'], unit='s')
    gdf_tooltip['TOI_formatted'] = gdf_tooltip['TOI_timedelta'].apply(
        lambda td: (
            f"{int(td.total_seconds() // 3600):02d} Hours, "
            f"{int((td.total_seconds() % 3600) // 60):02d} Minutes, "
            f"{int(td.total_seconds() % 60):02d} Seconds"
        ) if pd.notnull(td) else "00:00:00"
    )
    gdf_tooltip.drop(columns=['TOI_timedelta'], inplace=True)

    # 6. Build custom HTML tooltip content.
    def build_tooltip_html(row):
        keys_to_check = ['Player_Count', 'Games_Played', 'TOI_sec', 'G', 'A', 'Pts',
                         'GPP', 'GPG', 'APP', 'APG', 'PtsPP', 'PtsPG']
        if all(pd.isnull(row.get(key)) for key in keys_to_check):
            return ""
        total_players = int(row['Player_Count']) if pd.notnull(row['Player_Count']) else None
        total_games = int(row['Games_Played']) if pd.notnull(row['Games_Played']) else None
        toi = row.get('TOI_sec')
        total_players_str = str(total_players) if total_players is not None else "N/A"
        total_games_str = str(total_games) if total_games is not None else "N/A"
        toi_str = row['TOI_formatted'] if toi is not None else "N/A"
        top_scorer_line = re.sub(r'(\d+)\.0\b', r'\1', str(row['TopScorer_StatLine']))
        html = f"""
        <div style="font-family: 'Exo 2', sans-serif; font-size: 12px; background-color: white; padding: 5px;">
            <h4 style="margin: 0; text-align: center;">{row['State_Province']}</h4>
            <p style="text-align: center; margin: 5px 0; font-size: 12px;">
                <strong>Total Players:</strong> {total_players_str} &nbsp;&nbsp;
                <strong>Total Games Played:</strong> {total_games_str} &nbsp;&nbsp;<br>
                <strong>Total Time On Ice:</strong> {toi_str}
            </p>
            <table style="width: 100%; border-collapse: collapse; margin-top: 5px;">
                <thead>
                    <tr>
                        <th style="border: 1px solid #ddd; padding: 4px;">Stat</th>
                        <th style="border: 1px solid #ddd; padding: 4px;">Total</th>
                        <th style="border: 1px solid #ddd; padding: 4px;">Per Player</th>
                        <th style="border: 1px solid #ddd; padding: 4px;">Per Game</th>
                    </tr>
                </thead>
                <tbody>
                    <tr>
                        <td style="border: 1px solid #ddd; padding: 4px;">Goals</td>
                        <td style="border: 1px solid #ddd; padding: 4px;">{row['G']:.0f}</td>
                        <td style="border: 1px solid #ddd; padding: 4px;">{row['GPP']:.2f}</td>
                        <td style="border: 1px solid #ddd; padding: 4px;">{row['GPG']:.3f}</td>
                    </tr>
                    <tr>
                        <td style="border: 1px solid #ddd; padding: 4px;">Assists</td>
                        <td style="border: 1px solid #ddd; padding: 4px;">{row['A']:.0f}</td>
                        <td style="border: 1px solid #ddd; padding: 4px;">{row['APP']:.2f}</td>
                        <td style="border: 1px solid #ddd; padding: 4px;">{row['APG']:.3f}</td>
                    </tr>
                    <tr>
                        <td style="border: 1px solid #ddd; padding: 4px;">Points</td>
                        <td style="border: 1px solid #ddd; padding: 4px;">{row['Pts']:.0f}</td>
                        <td style="border: 1px solid #ddd; padding: 4px;">{row['PtsPP']:.2f}</td>
                        <td style="border: 1px solid #ddd; padding: 4px;">{row['PtsPG']:.3f}</td>
                    </tr>
                </tbody>
            </table>
            <p style="margin: 5px 0; text-align: center; font-size: 12px;">
                Top Returning Scorer: <strong>{row['TopScorer_Name']}</strong><br>
                {row['Hometown']} - <i>{row['Team']}</i><br>
                {top_scorer_line}
            </p>
        </div>
        """
        return html

    gdf_tooltip['tooltip_html'] = gdf_tooltip.apply(build_tooltip_html, axis=1)
    gdf_tooltip['tooltip_html'] = gdf_tooltip['tooltip_html'].apply(Markup)

    # 7. Create GeoJSON layer with rich HTML popups (trigger on click).
    gdf_tooltip_filtered = gdf_tooltip[gdf_tooltip['tooltip_html'].notnull()].copy()
    popup = folium.GeoJsonPopup(
        fields=["tooltip_html"],
        aliases=[""],
        localize=True,
        parse_html=True,
        max_width=300,
        sticky=False
    )
    popup_layer = folium.FeatureGroup(name="Show Detailed Popups (Click)", show=True)
    folium.GeoJson(
        gdf_tooltip_filtered.__geo_interface__,
        style_function=lambda feature: {
            'fillColor': 'transparent',
            'color': 'transparent',
            'weight': 0
        },
        popup=popup
    ).add_to(popup_layer)
    popup_layer.add_to(folium_map)

    # Add a click handler so that clicking on the map closes any open tooltip.
    close_on_click_js = f"""
    <script>
        var mapInstance = {folium_map.get_name()};
        mapInstance.on('click', function(e) {{
            mapInstance.closePopup();
        }});
    </script>
    """
    folium_map.get_root().html.add_child(folium.Element(close_on_click_js))



# ...

    # 8. Add a heatmap layer from merged_df coordinates.
    coords_df = _clean_coords(merged_df)

    print(f"[HeatMap] kept {len(coords_df)}/{len(merged_df)} rows with valid coords")

    heat_data = coords_df[['Latitude', 'Longitude']].to_numpy().tolist()
    heatmap_layer = folium.FeatureGroup(name='Heatmap', show=True)
    HeatMap(heat_data, radius=25, blur=15, max_intensity=20).add_to(heatmap_layer)
    heatmap_layer.add_to(folium_map)


   # 9. Add marker cluster layer for individual players with custom logo icons.
    cluster_group = folium.FeatureGroup(name='Individual Players', control=True, show=False)
    marker_cluster = MarkerCluster(
        spiderfy_on_max_zoom=True,
        show_coverage_on_hover=False,
        max_cluster_radius=20,
        disableClusteringAtZoom=14,
        animateAddingMarkers=True,
        zoomToBoundsOnClick=True
    ).add_to(cluster_group)

    for idx, row in coords_df.iterrows():  # << use coords_df, not merged_df
        team_name = row['Team']
        logo_info = school_info_df.loc[school_info_df['Team'] == team_name, 'logo_abv'].values
        if len(logo_info) == 0:
            continue

        logo_abv = logo_info[0]
        logo_path = os.path.join(logo_folder, f"{logo_abv}.png")
        if not os.path.exists(logo_path):
            continue

        logo_icon = CustomIcon(logo_path, icon_size=logo_size)
        player_count = row['Player_Count']
        current_index = row['city_group_index']

        # coords_df guarantees row['Latitude']/['Longitude'] are valid numbers
        base_lat = float(row['Latitude'])
        base_lon = float(row['Longitude'])

        # Offset only if >1 in that city group
        if pd.notna(player_count) and player_count > 1:
            try:
                lat_offset, lon_offset = add_circular_offset(base_lat, base_lon, int(player_count), int(current_index))
            except Exception:
                lat_offset, lon_offset = base_lat, base_lon
        else:
            lat_offset, lon_offset = base_lat, base_lon

        # Guard: if offset produced NaNs/None, skip
        if lat_offset is None or lon_offset is None:
            continue
        if pd.isna(lat_offset) or pd.isna(lon_offset):
            continue

        # (optional) final range check
        if not (-90 <= lat_offset <= 90 and -180 <= lon_offset <= 180):
            continue

        # --- your existing position/yr/draft/tooltip code unchanged ---
        position = row['Position']
        if position == "Forawards":
            position = "Forward"
        elif position == "Forwards":
            position = "Forward"
        elif position == "Defencemen":
            position = "Defenseman"
        elif position == "Goaltenders":
            position = "Goaltender"

        yr_mapping = {'Fr': 'Freshman', 'So': 'Sophomore', 'Jr': 'Junior', 'Sr': 'Senior'}
        yr = yr_mapping.get(row['Yr'], row['Yr'])

        if all([pd.notna(row[col]) and row[col] != "" for col in ['Draft_Year', 'NHL_Team', 'D_Round']]):
            draft_info = f"<span style='color: #333;'><strong>Drafted {row['Draft_Year']} - </strong></span> {row['NHL_Team']}: {row['D_Round']} round<br>"
        else:
            draft_info = ""

        tooltip_html = f"""
        <div style="font-family: 'Exo 2', sans-serif; font-size: 12px; background-color: white; padding: 5px;">
            <strong>{row['First_Name']} {row['Last_Name']}</strong> {row['Team']}<br>
            <span style="color: #333;">{row['Hometown']}</span><br>
            <span style="color: #333;">{yr} {position}</span><br>
            {"<div style='font-size: 12px; color: black; margin-top: 5px;'>24-25 SEASON:</style><br>" +
            f"{row['Games_Played']} GP, {row['G']} G, {row['A']} A, {row['Pts']} PTS, {row['PIM']} PIM</div>"
            if pd.notna(row['Games_Played']) else ""}</style>
            {draft_info}
        </div>
        """

        folium.Marker(
            location=[lat_offset, lon_offset],
            tooltip=folium.Tooltip(tooltip_html),
            icon=logo_icon
        ).add_to(marker_cluster)

    cluster_group.add_to(folium_map)


    # 10. Add layer control and custom CSS for styling.
    folium.LayerControl().add_to(folium_map)

    custom_css = """
    <style>
        .leaflet-control-layers {
            background-color: #fff;
            border: 2px solid #ccc;
            border-radius: 5px;
            padding: 10px;
            box-shadow: 0 0 10px rgba(0,0,0,0.2);
        }
        .leaflet-control-layers-list {
            font-size: 16px;
            line-height: 1.5;
        }
        .leaflet-control-layers input[type="radio"],
        .leaflet-control-layers input[type="checkbox"] {
            transform: scale(1.5);
            margin-right: 8px;
        }
        .leaflet-control-layers label {
            color: #333;
            font-family: 'Exo 2', sans-serif;
        }
    </style>
    """
    folium_map.get_root().html.add_child(folium.Element(custom_css))

    close_on_click_js = f"""
    <script>
        var mapInstance = {folium_map.get_name()};
        mapInstance.on('click', function(e) {{
            mapInstance.closeTooltip();
        }});
    </script>
    """
    folium_map.get_root().html.add_child(folium.Element(close_on_click_js))

    # Add a custom reset button in the top right corner.
    reset_view_js = f"""
    <script>
        var mapInstance = {folium_map.get_name()};
        var resetButton = L.control({{position: 'topright'}});
        resetButton.onAdd = function(map) {{
            var div = L.DomUtil.create('div', 'leaflet-control leaflet-bar leaflet-control-custom');
            div.innerHTML = '<button style="background: white; border: none; padding: 5px 10px; font-family: \\'Exo 2\\', sans-serif; cursor: pointer;">Reset Map</button>';
            L.DomEvent.disableClickPropagation(div);
            div.onclick = function() {{
                map.setView([{map_center[0]}, {map_center[1]}], {zoom_start});
            }};
            return div;
        }};
        resetButton.addTo(mapInstance);
    </script>
    """
    folium_map.get_root().html.add_child(folium.Element(reset_view_js))

    # Add a Note/Attribution box.
    attribution_html = """
    <div style="
        position: fixed;
        bottom: 20px;
        right: 20px;
        z-index: 9999;
        background-color: rgba(255, 255, 255, 0.8);
        padding: 10px;
        border: 1px solid #ccc;
        border-radius: 5px;
        font-family: 'Exo 2', sans-serif;
        font-size: 14px;
        color: #333;
    ">
        Map created with Folium and OpenStreetMap<br>
        Created by J.Smith<br>
        <span style="font-size: 11px;">Roster and Statistical Data: CollegeHockeyNews.com</span>
    </div>
    """
    folium_map.get_root().html.add_child(folium.Element(attribution_html))

    # Add Title Annotation Box.
    annotation_html = """
    <div style="
        position: fixed;
        top: 10px;
        left: 50px;
        z-index: 9999;
        background-color: rgba(255, 255, 255, 0.9);
        padding: 10px;
        border: 1px solid #ccc;
        border-radius: 5px;
        font-family: 'Exo 2', sans-serif;
        color: #333;
    ">
        <h2 style="margin: 0; font-size: 20px;">NCAA Hockey Hometown Atlas: 2025</h2>
        <p style="margin: 0; font-size: 14px; font-style: italic;">Mapping the Hometowns of all NCAA D1 Hockey Players</p>
    </div>
    """
    folium_map.get_root().html.add_child(folium.Element(annotation_html))

    return folium_map

# Build the map using your data and then save to an HTML file.
enhanced_player_map = create_map_with_team_logos(merged_df, school_info_df, logo_folder, gdf_states)
enhanced_map_file_path = os.path.join('..', 'TEMP', '2025_full_division_player_origin_map_with_stats_v2.html')
enhanced_player_map.save(enhanced_map_file_path)


C:\Users\jbanc\AppData\Local\Temp\ipykernel_10316\469878594.py:76: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  gdf_global = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
C:\Users\jbanc\AppData\Local\Temp\ipykernel_10316\469878594.py:110: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  territory_counts_gdf['centroid'] = territory_counts_gdf.geometry.centroid


[HeatMap] kept 1759/1763 rows with valid coords
